# Module 7.2: Serving Industrial ML Models via FastAPI REST Endpoints
### 🇵🇱 Serwowanie Modeli Przemysłowych za Pomocą Punktów Końcowych REST w FastAPI

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/max480425/industrial-ml-course/blob/main/public/notebooks/m07_l02_serving_models_fastapi_rest.ipynb)

**Industrial Machine Learning & Predictive Maintenance Interactive Laboratory**

> **Summary:** Bridge OT and IT by packaging trained machine learning pipelines into high-performance, asynchronous FastAPI microservices with Pydantic validation.  
> 🇵🇱 **Streszczenie:** Połącz świat OT i IT: wdrażaj potoki uczenia maszynowego jako asynchroniczne mikroserwisy FastAPI z walidacją typów w Pydantic.

- **Estimated Time:** 25 minutes
- **Runtime:** Python 3.10+ (CPU or Free Tier T4 GPU)
- **Packages:** `numpy`, `scipy`, `scikit-learn`, `pandas`, `matplotlib`, `seaborn`
---


In [ ]:
# Step 1: Install & Verify Python Scientific Environment
!pip install -q numpy scipy scikit-learn pandas matplotlib seaborn

import numpy as np
import pandas as pd
import scipy.signal as signal
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'monospace'
print("✓ Industrial ML environment loaded successfully.")


### Key Engineering Concepts:
1. **OT/IT Convergence via REST**: Connecting plant SCADA/MES systems to Python predictive maintenance pipelines through standardized JSON HTTP endpoints.
2. **Pydantic Validation**: Enforcing strict physical boundary checks on sensor telemetry (e.g. current must be $4.0 - 20.0	ext{ mA}$) before inference.
3. **Sub-5ms Latency Requirement**: PLC cycles require microsecond response times; asynchronous FastAPI servers handle continuous telemetry without blocking.


In [ ]:
# 1. Define Pydantic Telemetry Schema & Simulate FastAPI Microservice
import time

class MockTelemetryPayload:
    # Simulates Pydantic model validation on industrial inputs
    def __init__(self, machine_id, current_ma, pressure_bar, temp_c):
        self.machine_id = machine_id
        # Physical boundary checks
        if not (3.6 <= current_ma <= 22.0):
            raise ValueError(f"Sensor current out of physical bounds (4-20mA loop): {current_ma} mA")
        if not (0.0 <= pressure_bar <= 350.0):
            raise ValueError(f"Pressure out of physical transmitter bounds: {pressure_bar} bar")
        self.current_ma = current_ma
        self.pressure_bar = pressure_bar
        self.temp_c = temp_c

def mock_predict_endpoint(payload):
    t0 = time.perf_counter()
    # Fast decision logic
    is_anomaly = payload.current_ma > 18.5 or payload.pressure_bar > 290.0 or payload.temp_c > 85.0
    risk_score = 0.85 if is_anomaly else 0.05
    latency_ms = (time.perf_counter() - t0) * 1000
    return {
        "machine_id": payload.machine_id,
        "prediction": "ALERT" if is_anomaly else "NORMAL",
        "risk_score": risk_score,
        "latency_ms": latency_ms
    }

# Test valid telemetry payload
valid_sample = MockTelemetryPayload("PUMP_01", 14.5, 240.0, 52.0)
resp = mock_predict_endpoint(valid_sample)
print(f"FastAPI Response : {resp}")


In [ ]:
# 2. Benchmark Simulated SCADA REST Request Latency (1,000 Calls)
latencies = []
for _ in range(1000):
    t0 = time.perf_counter()
    # Mock payload parsing and inference
    _ = mock_predict_endpoint(valid_sample)
    latencies.append((time.perf_counter() - t0) * 1000)

latencies = np.array(latencies)
p99_latency = np.percentile(latencies, 99)

plt.figure(figsize=(10, 4.5))
plt.hist(latencies, bins=30, color='#0284c7', edgecolor='black', alpha=0.7)
plt.axvline(p99_latency, color='#ef4444', linestyle='--', label=f'99th Percentile = {p99_latency:.3f} ms')
plt.axvline(5.0, color='#f59e0b', linestyle=':', label='Industrial Limit (5.0 ms)')

plt.title("FastAPI Telemetry Inference Latency Distribution (1,000 SCADA calls)", fontweight='bold')
plt.xlabel("Request Latency [milliseconds]")
plt.ylabel("Call Frequency")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 📋 Practical Hands-On Checklist

- [ ] **Task 1:** Define strict Pydantic telemetry input schema with physical engineering bounds *(🇵🇱 Zdefiniuj ścisły schemat danych telemetrycznych w Pydantic z inżynierskimi limitami fizycznymi)*
- [ ] **Task 2:** Build asynchronous FastAPI web service with /health and /predict endpoints *(🇵🇱 Zbuduj asynchroniczny serwis FastAPI z punktami końcowymi /health i /predict)*
- [ ] **Task 3:** Benchmark sub-5ms HTTP inference response latency under simulated SCADA payload traffic *(🇵🇱 Zweryfikuj czas odpowiedzi inferencji poniżej 5ms przy symulowanym ruchu zbieżnym z SCADA)*


In [ ]:
# Exercise Challenge:
# Assert that Pydantic rejects an invalid current reading of 1.2 mA (Broken Wire)
rejected_flag = False
try:
    MockTelemetryPayload("PUMP_01", 1.2, 240.0, 52.0)
except ValueError:
    rejected_flag = True

assert rejected_flag, "Pydantic validation failed to catch broken wire input!"
assert p99_latency < 5.0, "Inference response latency exceeded 5ms limit!"
print("✓ FastAPI serving and input validation verified.")
